# Phase 3 - Data Profiling

**Goal:** produce a data-quality report before any cleaning. We inspect shape, dtypes,
missing values, duplicates, invalid values, dates, numeric ranges, categorical
consistency, ID consistency, and referential integrity across the 5 raw tables.

We DO NOT change data here. We only observe and record issues. Cleaning happens in Phase 4.

For every issue we find, we note: **what is wrong, why it matters, how to handle it, and why**.

## 1. Load the raw data
Raw files live in `data/raw/`. We read everything as-is (no parsing tricks) so we can see the raw state.

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

# Absolute path so it works no matter where Jupyter is launched from.
RAW = r'c:\Users\Sweta\OneDrive\Desktop\proj2\data\raw'
print('Reading raw data from:', RAW)

employees = pd.read_csv(os.path.join(RAW, 'employees.csv'))
trips = pd.read_csv(os.path.join(RAW, 'trips.csv'))
expenses = pd.read_csv(os.path.join(RAW, 'expense_line_items.csv'))
offices = pd.read_csv(os.path.join(RAW, 'offices.csv'))
vendors = pd.read_csv(os.path.join(RAW, 'vendors.csv'))

tables = {
    'employees': employees,
    'trips': trips,
    'expenses': expenses,
    'offices': offices,
    'vendors': vendors,
}
print('Loaded:', {k: v.shape for k, v in tables.items()})

## 2. Shape, dtypes, and a peek at each table
`.info()` shows column dtypes and non-null counts in one view. This is the fastest way to spot
columns pandas read as text that should be dates/numbers, and columns with missing values.

In [ ]:
for name, df in tables.items():
    print('=' * 70)
    print(name.upper(), '   shape =', df.shape)
    print('-' * 70)
    df.info()
    print()

In [ ]:
employees.head()

In [ ]:
trips.head()

In [ ]:
expenses.head()

## 3. Missing values
We count missing values per column and show them as a percentage. Missing data matters because:
- a missing `vendor_id` on a trip means we cannot judge preferred vs non-preferred for that trip;
- a blank `exception_reason` is EXPECTED when `is_policy_exception` is False (not a real problem);
- a blank `contract_start_date` is EXPECTED for non-preferred vendors.

So we do not treat all missingness the same - we interpret it.

In [ ]:
def missing_report(df, name):
    miss = df.isna().sum()
    pct = (miss / len(df) * 100).round(2)
    out = pd.DataFrame({'missing': miss, 'missing_pct': pct})
    out = out[out['missing'] > 0].sort_values('missing', ascending=False)
    print('=' * 60)
    print(name.upper(), '- columns with missing values')
    if out.empty:
        print('  (none)')
    else:
        print(out.to_string())
    print()

for name, df in tables.items():
    missing_report(df, name)

## 4. Duplicate checks
Two kinds of duplicates matter:
1. **Duplicate primary keys** - would break joins and double-count. Should be zero.
2. **Fully duplicated rows** - same record entered twice.

In [ ]:
pk_map = {
    'employees': 'employee_id',
    'trips': 'trip_id',
    'expenses': 'line_item_id',
    'offices': 'office_id',
    'vendors': 'vendor_id',
}

for name, df in tables.items():
    pk = pk_map[name]
    dup_pk = df[pk].duplicated().sum()
    dup_rows = df.duplicated().sum()
    print(f'{name:12s} duplicate {pk} = {dup_pk:>4}   fully-duplicated rows = {dup_rows}')

## 5. Categorical value checks
We list the distinct values of the key categorical columns. We are looking for:
- unexpected categories (typos, casing differences, stray whitespace);
- values that should be booleans but were read as text.

In [ ]:
cat_cols = {
    'employees': ['department', 'job_level', 'employment_status'],
    'trips': ['trip_purpose', 'booking_channel', 'trip_status'],
    'expenses': ['category', 'currency', 'approval_status', 'is_policy_exception', 'approved_by_level'],
    'offices': ['region', 'country'],
    'vendors': ['vendor_type', 'is_preferred_vendor'],
}

for name, cols in cat_cols.items():
    df = tables[name]
    print('=' * 60)
    print(name.upper())
    for c in cols:
        vals = df[c].value_counts(dropna=False)
        print(f'\n  {c}  ({df[c].nunique(dropna=True)} distinct):')
        print(vals.to_string().replace('\n', '\n    '))
    print()

## 6. Whitespace / text-hygiene check
Leading or trailing spaces create fake categories (e.g. 'hotel' vs 'hotel '). We check the
object (text) columns for values that change when stripped.

In [ ]:
def whitespace_check(df, name):
    hits = []
    for c in df.select_dtypes(include='object').columns:
        s = df[c].dropna().astype(str)
        n_ws = (s != s.str.strip()).sum()
        if n_ws > 0:
            hits.append((c, int(n_ws)))
    print(f'{name}:', hits if hits else 'no leading/trailing whitespace')

for name, df in tables.items():
    whitespace_check(df, name)

## 7. Numeric range checks (expense amounts)
`amount` drives every financial figure. We check for negatives, zeros, and extreme outliers.
We use percentiles (not a statistical test) - simple and explainable.

In [ ]:
amt = expenses['amount']
print('amount describe():')
print(amt.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2).to_string())
print()
print('negative amounts :', int((amt < 0).sum()))
print('zero amounts     :', int((amt == 0).sum()))
print('amounts > 5000   :', int((amt > 5000).sum()))
print('max amount       :', round(amt.max(), 2))

## 8. Date validation
We parse the date columns and check:
- can they all be parsed (any junk values)?
- on trips: is `trip_end_date` ever BEFORE `trip_start_date`? (impossible - a real data issue)
- is `booked_date` ever AFTER `trip_start_date`? (booked after travel began - suspicious)

In [ ]:
trip_dates = ['booked_date', 'trip_start_date', 'trip_end_date']
t = trips.copy()
for c in trip_dates:
    t[c] = pd.to_datetime(t[c], errors='coerce')
    n_bad = t[c].isna().sum() - trips[c].isna().sum()
    print(f'{c}: unparseable values = {int(n_bad)}')

end_before_start = (t['trip_end_date'] < t['trip_start_date']).sum()
booked_after_start = (t['booked_date'] > t['trip_start_date']).sum()
print()
print('trips with trip_end_date < trip_start_date :', int(end_before_start))
print('trips with booked_date > trip_start_date   :', int(booked_after_start))
print()
print('Examples of end < start:')
display(t.loc[t['trip_end_date'] < t['trip_start_date'],
              ['trip_id', 'booked_date', 'trip_start_date', 'trip_end_date', 'trip_status']].head(10))

In [ ]:
# expense submitted_date parse check
sd = pd.to_datetime(expenses['submitted_date'], errors='coerce')
print('expense submitted_date unparseable :', int(sd.isna().sum() - expenses['submitted_date'].isna().sum()))
print('submitted_date range :', sd.min(), '->', sd.max())

## 9. Referential integrity (do the foreign keys line up?)
This is critical because our analysis joins these tables. We check each FK points to a real PK:
- every `expenses.trip_id` exists in `trips`?
- every `trips.employee_id` exists in `employees`?
- every `employees.office_id` exists in `offices`?
- every non-null `trips.vendor_id` exists in `vendors`?
- do any trips have NO expense lines (so no derivable cost)?

In [ ]:
# expenses.trip_id -> trips.trip_id
orphan_exp = ~expenses['trip_id'].isin(trips['trip_id'])
print('expense lines whose trip_id is NOT in trips :', int(orphan_exp.sum()))

# trips.employee_id -> employees.employee_id
orphan_trip_emp = ~trips['employee_id'].isin(employees['employee_id'])
print('trips whose employee_id is NOT in employees :', int(orphan_trip_emp.sum()))

# employees.office_id -> offices.office_id
orphan_emp_off = ~employees['office_id'].isin(offices['office_id'])
print('employees whose office_id is NOT in offices :', int(orphan_emp_off.sum()))

# trips.vendor_id -> vendors.vendor_id (ignoring missing vendor_id)
known_vendor = trips['vendor_id'].dropna()
orphan_trip_vendor = ~known_vendor.isin(vendors['vendor_id'])
print('trips whose (non-null) vendor_id is NOT in vendors :', int(orphan_trip_vendor.sum()))

# trips with no expense lines
trips_with_expense = trips['trip_id'].isin(expenses['trip_id'])
print('trips with NO expense lines (no derivable cost)   :', int((~trips_with_expense).sum()))

## 10. Trip status distribution and vendor_id missingness
Two numbers we will rely on heavily later, so we surface them now.

In [ ]:
print('trip_status:')
print(trips['trip_status'].value_counts(dropna=False).to_string())
print()
n_missing_vendor = trips['vendor_id'].isna().sum()
print(f'trips with missing vendor_id : {int(n_missing_vendor)} '
      f'({n_missing_vendor / len(trips) * 100:.1f}% of all trips)')

## 11. Data-quality summary
Run this last. It collects the headline numbers into one small table you can screenshot for
the README / interview. Interpretation and treatment decisions are written up in
`docs/03_data_quality_report.md`.

In [ ]:
summary = {
    'employees_rows': len(employees),
    'trips_rows': len(trips),
    'expense_rows': len(expenses),
    'offices_rows': len(offices),
    'vendors_rows': len(vendors),
    'trips_missing_vendor_id': int(trips['vendor_id'].isna().sum()),
    'trips_end_before_start': int(end_before_start),
    'trips_booked_after_start': int(booked_after_start),
    'trips_with_no_expenses': int((~trips_with_expense).sum()),
    'orphan_expense_trip_id': int(orphan_exp.sum()),
    'expense_negative_amount': int((amt < 0).sum()),
    'expense_zero_amount': int((amt == 0).sum()),
    'expense_rejected_lines': int((expenses['approval_status'] == 'rejected').sum()),
    'expense_pending_lines': int((expenses['approval_status'] == 'pending').sum()),
    'expense_policy_exceptions': int((expenses['is_policy_exception'] == True).sum()),
}
summary_df = pd.DataFrame.from_dict(summary, orient='index', columns=['value'])
summary_df